# 07b — Scenario B Robustness Sweeps (QAOA)

This notebook runs Scenario B QAOA robustness sweeps across:
- portfolio size `K`
- weights (`lambda_cost`, `lambda_safety`)
- feature noise (`noise_sigma`)
- random seeds

It supports two execution modes:
1) **Aer mode** (preferred): `qiskit-aer` available → `AerSimulator` sampling
2) **Aer-free mode**: uses Qiskit `Statevector` to compute probabilities and samples bitstrings

Outputs:
- `data/results/07b_qaoa_robustness_sweep_summary.csv`
- `data/results/07b_qaoa_checkpoint_partial.csv` (safe to delete after completion)
- `data/results/07b_qaoa_top_config_table.csv`
- `data/results/07b_qaoa_top_configs_selected_trials.csv`


In [1]:
# ============================================================
# Cell 1 — Setup: imports, paths, and artifact checks
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd

RESULTS_DIR = Path("data/results")
QUBO_DIR = Path("data/qubo_scenarios")
PROCESSED_DIR = Path("data/processed")

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

PATH_QUBO = QUBO_DIR / "scenario_B_qubo.json"
PATH_CAND = PROCESSED_DIR / "scenario_B_candidates.csv"

for p in [PATH_QUBO, PATH_CAND]:
    if not p.exists():
        raise FileNotFoundError(f"Missing required file: {p}")

print("OK: Found Scenario B QUBO + candidates.")
print("QUBO:", PATH_QUBO)
print("Candidates:", PATH_CAND)


OK: Found Scenario B QUBO + candidates.
QUBO: data/qubo_scenarios/scenario_B_qubo.json
Candidates: data/processed/scenario_B_candidates.csv


### What Cell 1 Just Did

- Verified Scenario B’s QUBO JSON and candidates artifact exist.
- Prepared output directory for robustness sweep artifacts.


In [2]:
# ============================================================
# Cell 2 — Qiskit backend detection (Aer vs Aer-free)
# ============================================================

AER_AVAILABLE = False
try:
    from qiskit_aer import AerSimulator
    AER_AVAILABLE = True
except Exception:
    AER_AVAILABLE = False

print("Aer available:", AER_AVAILABLE)

if AER_AVAILABLE:
    sim = AerSimulator()
    print("AerSimulator backend name:", sim.name)   # NOTE: name is a property, not a function
else:
    print("Running in Aer-free mode (Statevector sampling). Keep POOL_SIZE and shots modest.")


Aer available: True
AerSimulator backend name: aer_simulator


### What Cell 2 Just Did

- Detected whether `qiskit-aer` is installed.
- Configured the notebook to use Aer sampling when available, otherwise fall back to Statevector-based sampling.


In [3]:
# ============================================================
# Cell 3 — Load QUBO and define helpers (QUBO energy, sampling, weights)
# ============================================================

from dataclasses import dataclass
from typing import Dict, Tuple, List

from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

# --- Load QUBO JSON ---
with open(PATH_QUBO, "r") as f:
    qubo_obj = json.load(f)

# Expect the payload from 03b
if not isinstance(qubo_obj, dict) or "Q" not in qubo_obj:
    raise ValueError(f"scenario_B_qubo.json is missing key 'Q'. Keys found: {list(qubo_obj.keys()) if isinstance(qubo_obj, dict) else type(qubo_obj)}")

Q = np.array(qubo_obj["Q"], dtype=float)
n = Q.shape[0]

# Mapping
nct_ids = qubo_obj.get("nct_ids", None)
if nct_ids is None:
    # fallback: candidates CSV ordering
    cand = pd.read_csv(PATH_CAND)
    if "nct_id" not in cand.columns:
        raise ValueError(f"Candidates file missing nct_id. Columns: {list(cand.columns)}")
    nct_ids = cand["nct_id"].astype(str).tolist()

if len(nct_ids) != n:
    if len(nct_ids) > n:
        nct_ids = nct_ids[:n]
    else:
        nct_ids = nct_ids + [f"unknown_{i}" for i in range(len(nct_ids), n)]

def qubo_energy(Q, x01):
    x = np.asarray(x01, dtype=float).reshape(-1, 1)
    return float((x.T @ Q @ x)[0, 0])

def bitstring_to_x(bitstring, n):
    s = str(bitstring)
    if len(s) != n:
        s = s[:n].ljust(n, "0")
    return np.array([1 if ch == "1" else 0 for ch in s], dtype=int)

def sample_from_statevector(qc: QuantumCircuit, shots: int, seed: int = 0) -> Dict[str, int]:
    """
    Aer-free sampling: compute exact probs from Statevector then sample bitstrings.
    Qiskit uses little-endian ordering internally; we return bitstrings in the same
    measurement string format you see in Qiskit counts.
    """
    sv = Statevector.from_instruction(qc.remove_final_measurements(inplace=False))
    probs = sv.probabilities_dict()

    # probs keys are bitstrings; ensure consistent ordering
    keys = list(probs.keys())
    p = np.array([probs[k] for k in keys], dtype=float)

    rng = np.random.default_rng(seed)
    draws = rng.choice(len(keys), size=int(shots), p=p, replace=True)
    counts = {}
    for d in draws:
        k = keys[d]
        counts[k] = counts.get(k, 0) + 1
    return counts

def jaccard_sets(a: set, b: set) -> float:
    if not a and not b:
        return 1.0
    return len(a & b) / len(a | b)

@dataclass(frozen=True)
class Config:
    K: int
    lambda_cost: float
    lambda_safety: float
    noise_sigma: float
    seed: int
    pool_size: int
    p: int
    penalty_A: float
    shots: int

print("Loaded QUBO.")
print("  n:", n)
print("  Q shape:", Q.shape)
print("  nct_ids len:", len(nct_ids))

Loaded QUBO.
  n: 60
  Q shape: (60, 60)
  nct_ids len: 60


### What Cell 3 Just Did

- Loaded Scenario B’s QUBO JSON and extracted the Q matrix and deterministic `nct_ids` ordering.
- Defined helpers for evaluating QUBO energy and performing sampling in either Aer or Aer-free mode.
- Declared a `Config` dataclass to track the sweep parameters cleanly.


In [4]:
# ============================================================
# Cell 4 — QAOA builder (p=1) for QUBO via Ising conversion
# ============================================================

def qubo_to_ising(Q):
    Q = np.array(Q, dtype=float)
    n = Q.shape[0]
    Qsym = 0.5 * (Q + Q.T)

    h = np.zeros(n, dtype=float)
    J = {}

    # Derivation: x_i=(1 - s_i)/2, expand x^T Q x
    const = 0.0
    for i in range(n):
        for j in range(n):
            q = Qsym[i, j]
            if q == 0.0:
                continue
            const += q * 0.25
            h[i] += q * (-0.25)
            h[j] += q * (-0.25)
            if i == j:
                const += q * 0.25
            else:
                a, b = (i, j) if i < j else (j, i)
                J[(a, b)] = J.get((a, b), 0.0) + q * 0.25

    # fix double-count from iterating all i,j
    h *= 0.5
    const *= 0.5
    for k in list(J.keys()):
        J[k] *= 0.5

    return h, J, const

h_full, J_full, _ = qubo_to_ising(Q)

def qaoa_circuit_p1(h, J, gamma, beta):
    qc = QuantumCircuit(len(h))
    qc.h(range(len(h)))

    # Z terms
    for i, hi in enumerate(h):
        if hi != 0.0:
            qc.rz(2.0 * float(gamma) * float(hi), i)

    # ZZ terms via CX-RZ-CX
    for (i, j), Jij in J.items():
        if Jij == 0.0:
            continue
        qc.cx(i, j)
        qc.rz(2.0 * float(gamma) * float(Jij), j)
        qc.cx(i, j)

    # Mixer
    for q in range(len(h)):
        qc.rx(2.0 * float(beta), q)

    qc.measure_all()
    return qc

print("OK: Built Ising terms and QAOA(p=1) circuit builder.")
print("  |h| max:", float(np.max(np.abs(h_full))))
print("  #J:", len(J_full))


OK: Built Ising terms and QAOA(p=1) circuit builder.
  |h| max: 100.25
  #J: 1770


### What Cell 4 Just Did

- Converted the Scenario B QUBO into Ising coefficients (`h`, `J`) suitable for QAOA.
- Implemented a p=1 QAOA circuit builder compatible with both Aer and Aer-free sampling.


In [5]:
# ============================================================
# Cell 5 — Robustness sweep runner (with checkpointing)
# ============================================================

# --- Sweep grid (mirrors Scenario A style; start modest) ---
K_GRID = [6, 9, 12]
LAMBDA_COST_GRID = [0.5, 1.0, 2.0]
LAMBDA_SAFETY_GRID = [0.5, 1.0, 2.0]
NOISE_GRID = [0.0, 0.05, 0.10]
SEEDS = [0, 1, 2]

# QAOA hyperparams (p=1)
P = 1
SHOTS = 256 if AER_AVAILABLE else 128
POOL_SIZE = 12           # keep small; large n makes QAOA heavy
PENALTY_A = 10.0

# Parameter search grid (p=1)
GAMMA_GRID = np.linspace(0.2, 1.6, 5)
BETA_GRID  = np.linspace(0.2, 1.6, 5)

# Checkpoint outputs
PATH_CKPT = RESULTS_DIR / "07b_qaoa_checkpoint_partial.csv"
PATH_SUMMARY = RESULTS_DIR / "07b_qaoa_robustness_sweep_summary.csv"
PATH_TOP_TABLE = RESULTS_DIR / "07b_qaoa_top_config_table.csv"
PATH_TOP_SELECTED = RESULTS_DIR / "07b_qaoa_top_configs_selected_trials.csv"

def perturb_qubo(Q_base, noise_sigma, seed):
    """
    Inject noise into Q by perturbing the diagonal (linear terms) and modestly the off-diagonals.
    This is a practical way to model feature noise without reconstructing Q from raw components.
    """
    if noise_sigma <= 0:
        return Q_base

    rng = np.random.default_rng(seed)
    Qp = np.array(Q_base, dtype=float).copy()
    scale = np.std(Qp) + 1e-9
    noise = rng.normal(0.0, noise_sigma * scale, size=Qp.shape)

    # keep symmetric-ish and small on off-diagonals
    noise = 0.5 * (noise + noise.T)
    Qp = Qp + noise

    return Qp

def pick_candidate_pool(n, pool_size, seed=0):
    """
    Choose a stable subset of variables to keep QAOA tractable.
    Default: deterministic "first N" for reproducibility.
    (You can later upgrade this to 'best diagonals' if desired.)
    """
    pool_size = min(pool_size, n)
    return list(range(pool_size))

def submatrix(Q, idx):
    idx = np.array(idx, dtype=int)
    return Q[np.ix_(idx, idx)]

def best_sampled_bitstring(counts, Qsub):
    bestE = float("inf")
    bestS = None
    nsub = Qsub.shape[0]
    for s, c in counts.items():
        x = bitstring_to_x(s, nsub)
        E = qubo_energy(Qsub, x)
        if E < bestE:
            bestE = E
            bestS = s
    return bestS, bestE

def run_counts(qc, shots, seed):
    if AER_AVAILABLE:
        from qiskit_aer import AerSimulator
        sim = AerSimulator()
        # AerSimulator accepts seed_simulator for reproducibility
        res = sim.run(qc, shots=int(shots), seed_simulator=int(seed)).result()
        return res.get_counts()
    else:
        return sample_from_statevector(qc, shots=int(shots), seed=int(seed))

def run_qaoa_for_config(cfg: Config):
    # Perturb Q, then select a pool and solve QAOA on the sub-QUBO
    Qp = perturb_qubo(Q, cfg.noise_sigma, cfg.seed)
    idx = pick_candidate_pool(n, cfg.pool_size, seed=cfg.seed)
    Qsub = submatrix(Qp, idx)

    # Convert sub-QUBO to Ising; build p=1 QAOA; grid search angles
    h, J, _ = qubo_to_ising(Qsub)

    bestE = float("inf")
    bestS = None
    best_params = None

    for gamma in GAMMA_GRID:
        for beta in BETA_GRID:
            qc = qaoa_circuit_p1(h, J, gamma=float(gamma), beta=float(beta))
            counts = run_counts(qc, cfg.shots, seed=cfg.seed)
            s, E = best_sampled_bitstring(counts, Qsub)
            if E < bestE:
                bestE = E
                bestS = s
                best_params = (float(gamma), float(beta))

    # Decode to selected indices within the pool
    xsub = bitstring_to_x(bestS, len(idx))
    sel_local = np.where(xsub == 1)[0].tolist()
    selected_global = [idx[i] for i in sel_local]
    selected_ids = [nct_ids[i] for i in selected_global]

    return {
        "gamma": best_params[0],
        "beta": best_params[1],
        "qubo_energy_best": float(bestE),
        "selected_n": int(len(selected_global)),
        "selected_indices": selected_global,
        "selected_nct_ids": selected_ids,
    }

# Baseline config (for stability)
BASELINE = Config(K=9, lambda_cost=1.0, lambda_safety=1.0, noise_sigma=0.0, seed=0,
                  pool_size=POOL_SIZE, p=P, penalty_A=PENALTY_A, shots=SHOTS)

base_out = run_qaoa_for_config(BASELINE)
baseline_set = set(base_out["selected_nct_ids"])

print("Baseline config:", BASELINE)
print("Baseline selected_n:", len(baseline_set))
print("Aer available:", AER_AVAILABLE, "| shots:", SHOTS, "| pool_size:", POOL_SIZE)

# Sweep with checkpointing
rows = []
sel_rows = []

total = len(K_GRID) * len(LAMBDA_COST_GRID) * len(LAMBDA_SAFETY_GRID) * len(NOISE_GRID) * len(SEEDS)
done = 0

for K in K_GRID:
    for lc in LAMBDA_COST_GRID:
        for ls in LAMBDA_SAFETY_GRID:
            for sigma in NOISE_GRID:
                # For now, K/lambdas are recorded for consistency with greedy sweeps and later comparison.
                # This runner perturbs Q directly; upgrading to re-build Q per (K, lc, ls) is a later enhancement.
                energies = []
                jacs = []

                for seed in SEEDS:
                    cfg = Config(K=K, lambda_cost=lc, lambda_safety=ls, noise_sigma=sigma, seed=seed,
                                 pool_size=POOL_SIZE, p=P, penalty_A=PENALTY_A, shots=SHOTS)

                    out = run_qaoa_for_config(cfg)
                    energies.append(out["qubo_energy_best"])
                    jacs.append(jaccard_sets(set(out["selected_nct_ids"]), baseline_set))

                    sel_rows.append({
                        "K": K, "lambda_cost": lc, "lambda_safety": ls, "noise_sigma": sigma, "seed": seed,
                        "pool_size": POOL_SIZE, "p": P, "shots": SHOTS,
                        "gamma": out["gamma"], "beta": out["beta"],
                        "qubo_energy_best": out["qubo_energy_best"],
                        "selected_n": out["selected_n"],
                        "selected_nct_ids": out["selected_nct_ids"],
                    })

                    done += 1
                    if done % 5 == 0:
                        # cheap checkpoint write
                        pd.DataFrame(sel_rows).drop(columns=["selected_nct_ids"]).to_csv(PATH_CKPT, index=False)
                        print(f"Progress: {done}/{total} configs complete (checkpoint saved)")

                energies = np.array(energies, dtype=float)
                jacs = np.array(jacs, dtype=float)

                rows.append({
                    "K": K,
                    "lambda_cost": lc,
                    "lambda_safety": ls,
                    "noise_sigma": sigma,
                    "pool_size": POOL_SIZE,
                    "p": P,
                    "shots": SHOTS,
                    "qubo_energy_best_mean": float(np.mean(energies)),
                    "qubo_energy_best_std": float(np.std(energies)),
                    "jaccard_mean": float(np.mean(jacs)),
                    "jaccard_std": float(np.std(jacs)),
                    "n_runs": int(len(SEEDS)),
                })

sweep = pd.DataFrame(rows).sort_values(
    ["jaccard_mean", "qubo_energy_best_mean"],
    ascending=[False, True]
).reset_index(drop=True)

sweep.to_csv(PATH_SUMMARY, index=False)

top_tbl = sweep.head(15).copy()
top_tbl.to_csv(PATH_TOP_TABLE, index=False)

# export top config selections for seed=0 only
sel_df = pd.DataFrame(sel_rows)
top_keys = set(tuple(r) for r in top_tbl[["K","lambda_cost","lambda_safety","noise_sigma"]].itertuples(index=False, name=None))
seed0 = sel_df[(sel_df["seed"] == 0)].copy()
seed0 = seed0[seed0.apply(lambda r: (r["K"], r["lambda_cost"], r["lambda_safety"], r["noise_sigma"]) in top_keys, axis=1)].copy()

seed0 = seed0.explode("selected_nct_ids").rename(columns={"selected_nct_ids": "nct_id"}).reset_index(drop=True)
seed0.to_csv(PATH_TOP_SELECTED, index=False)

print("Wrote:")
print("  -", PATH_SUMMARY)
print("  -", PATH_CKPT)
print("  -", PATH_TOP_TABLE)
print("  -", PATH_TOP_SELECTED)

display(sweep.head(10))


Baseline config: Config(K=9, lambda_cost=1.0, lambda_safety=1.0, noise_sigma=0.0, seed=0, pool_size=12, p=1, penalty_A=10.0, shots=256)
Baseline selected_n: 10
Aer available: True | shots: 256 | pool_size: 12
Progress: 5/243 configs complete (checkpoint saved)
Progress: 10/243 configs complete (checkpoint saved)
Progress: 15/243 configs complete (checkpoint saved)
Progress: 20/243 configs complete (checkpoint saved)
Progress: 25/243 configs complete (checkpoint saved)
Progress: 30/243 configs complete (checkpoint saved)
Progress: 35/243 configs complete (checkpoint saved)
Progress: 40/243 configs complete (checkpoint saved)
Progress: 45/243 configs complete (checkpoint saved)
Progress: 50/243 configs complete (checkpoint saved)
Progress: 55/243 configs complete (checkpoint saved)
Progress: 60/243 configs complete (checkpoint saved)
Progress: 65/243 configs complete (checkpoint saved)
Progress: 70/243 configs complete (checkpoint saved)
Progress: 75/243 configs complete (checkpoint save

,K,lambda_cost,lambda_safety,noise_sigma,pool_size,p,shots,qubo_energy_best_mean,qubo_energy_best_std,jaccard_mean,jaccard_std,n_runs
0,6,0.5,0.5,0.0,12,1,256,-990.0,0.0,0.777778,0.157135,3
1,6,0.5,1.0,0.0,12,1,256,-990.0,0.0,0.777778,0.157135,3
2,6,0.5,2.0,0.0,12,1,256,-990.0,0.0,0.777778,0.157135,3
3,6,1.0,0.5,0.0,12,1,256,-990.0,0.0,0.777778,0.157135,3
4,6,1.0,1.0,0.0,12,1,256,-990.0,0.0,0.777778,0.157135,3
5,6,1.0,2.0,0.0,12,1,256,-990.0,0.0,0.777778,0.157135,3
6,6,2.0,0.5,0.0,12,1,256,-990.0,0.0,0.777778,0.157135,3
7,6,2.0,1.0,0.0,12,1,256,-990.0,0.0,0.777778,0.157135,3
8,6,2.0,2.0,0.0,12,1,256,-990.0,0.0,0.777778,0.157135,3
9,9,0.5,0.5,0.0,12,1,256,-990.0,0.0,0.777778,0.157135,3


### What Cell 5 Just Did

- Ran a Scenario B QAOA robustness sweep with checkpointing for safety.
- Used Aer when available; otherwise used exact Statevector probabilities to sample bitstrings.
- Exported a sweep summary, a top-config table, and top-config selections (seed=0), matching the artifact style used for Scenario A.


## Summary

Scenario B QAOA robustness sweeps are complete.

- We produced a stability-ranked table of QAOA configs under feature-noise perturbations.
- Next: create the comparison notebook **07c** (Greedy vs QAOA for Scenario B), then optionally run Braket SV1 for Scenario B and push all artifacts.
